# Model Evaluation

Load `artifacts/evaluation_report.json` and optional `artifacts/training_metrics.json`.

In [ ]:
import json
from pathlib import Path

ROOT = Path("..").resolve() if (Path("..") / "artifacts").exists() else Path(".").resolve()
REPORT_PATH = ROOT / "artifacts" / "evaluation_report.json"
METRICS_PATH = ROOT / "artifacts" / "training_metrics.json"

In [ ]:
if REPORT_PATH.exists():
    report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
    print(f"Loaded {REPORT_PATH}")
else:
    report = None
    print("No evaluation report found.")
    print("Run: USE_MOCK_LLM=1 python fine_tuning/evaluate.py --max-samples 2")

In [ ]:
if report:
    print("ROUGE-L means:", report.get("rouge_l", {}))
    print("Mock mode:", report.get("mock_mode"))
    print("Evaluated samples:", report.get("evaluated_samples"))
    print("Checklist scores:")
    for model, scores in report.get("checklist_scores", {}).items():
        print(f"  {model}: {scores}")

In [ ]:
if report and report.get("samples"):
    import pandas as pd

    df = pd.DataFrame(
        [
            {
                "instruction": s["instruction"][:60] + "...",
                "rouge_l_base": s.get("rouge_l_base"),
                "rouge_l_finetuned": s.get("rouge_l_finetuned"),
            }
            for s in report["samples"]
        ]
    )
    df

In [ ]:
if METRICS_PATH.exists():
    training_metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
    epochs = training_metrics.get("epochs", [])
    if epochs:
        import matplotlib.pyplot as plt

        xs = [e["epoch"] for e in epochs]
        train_loss = [e.get("train_loss") for e in epochs]
        eval_loss = [e.get("eval_loss") for e in epochs]
        plt.plot(xs, train_loss, marker="o", label="train_loss")
        plt.plot(xs, eval_loss, marker="s", label="eval_loss")
        plt.xlabel("epoch")
        plt.ylabel("loss")
        plt.legend()
        plt.title("Training metrics")
        plt.show()
    else:
        print("No epoch metrics (training may have been skipped):", training_metrics.get("reason"))
else:
    print("No training_metrics.json — run fine_tuning/train.py on GPU first.")